In [0]:
# Silver layer: logs (web access log events)
# Source: b2b_ecommerce.bronze.logs (append-only, one raw log_line per row)
# Grain:  one row per distinct (source_file, log_line) pair (hashed to log_id)
# Approach: each raw line is parsed from Apache/Nginx combined format into
#           typed columns, cleaned ('-' placeholder -> NULL), standardized,
#           and MERGEd into b2b_ecommerce.silver.logs with AUTO CDC keyed by
#           log_id (SCD Type 1 = deduplication; events are immutable, so no
#           version history is kept).
#           Rows failing the drop-level data quality rules are NOT lost:
#           they are captured in b2b_ecommerce.quarantine.logs_silver
#           (dedicated quarantine schema, <table>_<layer> naming) with the
#           violated rule names in failed_checks.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_LOGS = "b2b_ecommerce.bronze.logs"
# Pipeline dataset (silver.customers, SCD Type 2) - governed customer reference
SILVER_CUSTOMERS = "customers"

# One tolerant head pattern covers the line variants verified in bronze
# (541,392 rows profiled 2026-11-12; the who/when/what/result groups are
# shared by every variant):
#   1. strict combined format:  IP i u [ts] "GET /path HTTP/2.0" 200 bytes "ref" "ua"
#   2. unquoted request (472):  IP i u [ts] GET /path HTTP/1.1 404 bytes ref UA-with-spaces
#   3. truncated ref/UA (243): IP i u [ts] "GET /path HTTP/2.0" 200 bytes "ref" "partial-UA
#   4. no bytes/tail (2):       IP i u [ts] "GET /path HTTP/2.0" 200
# plus 406 complete lines with ' |EXTRA|' junk appended (suffix stripped
# before parsing). Lines missing any of the 4 essentials (blank lines,
# '-- GARBAGE --', fragments, cut off before the request or status: 1,072
# rows) don't match HEAD_RE and are dropped by the expectations below.
# Limitation: a user_agent containing an escaped quote (\") will not match;
# such lines are counted with the other unparsable drops.
HEAD_RE = (
    r"^(\S+) (\S+) (\S+) \[([^\]]+)\] \"?([A-Z]+) (\S+) (HTTP/[0-9.]+)\"? "
    r"(\d{3})( (\S+)(.*))?$"
)
# HEAD_RE groups: 1 client_ip, 3 auth_user, 4 ts_raw, 5 method, 6
# req_target, 7 protocol, 8 status, 10 bytes, 11 free-form tail
# (' "referer" "ua"' or ' referer UA-with-spaces')


def extract(name, group):
    """Field from HEAD_RE on the junk-stripped line. regexp_extract returns
    an empty string (not NULL) when the regex does not match, so map
    no-match to NULL: unparsable lines then have NULL fields (caught by the
    not-null expectations) and are safe to cast (no ANSI cast of "" to int /
    timestamp)."""
    e = F.regexp_extract("stripped", HEAD_RE, group)
    return F.when(e != "", e).alias(name)


def null_if_absent(col):
    """'' (regex no-match), NULL, and '-' (access-log 'no value') -> NULL."""
    return F.when(col.isNull() | (col == "") | (col == "-"), None).otherwise(col)


def tail_referer(tail):
    """Quoted tail ' "referer" ...' -> referer; unquoted tail ' ref UA...'
    -> first token."""
    return (
        F.when(tail.startswith(' "'), F.regexp_extract(tail, r'^ "([^"]*)"', 1))
        .otherwise(F.regexp_extract(tail, r'^ (\S+)', 1))
    )


def tail_user_agent(tail):
    """Quoted tail: text between the referer quote and the (optional)
    closing quote; unquoted tail: everything after the first token.
    Truncated UAs keep their partial text; a truncated referer yields a
    NULL referer."""
    quoted = F.regexp_replace(F.regexp_replace(tail, r'^ "[^"]*"', ""), r'^ ?"?', "")
    quoted = F.regexp_replace(quoted, r'"$', "")
    unquoted = F.regexp_replace(tail, r'^ \S+ ?', "")
    return F.when(tail.startswith(' "'), quoted).otherwise(unquoted)


# ---------------------------------------------------------------------------
# 1) Parsed + cleaned + standardized stream of bronze.logs
# ---------------------------------------------------------------------------
def cleaned_logs(bronze):
    """Parse + clean + standardize raw bronze log rows (steps 1-2 only: no
    expectations, no referential join). Single source of truth shared by
    logs_clean_stream (the silver path) and the quarantine table below, so
    the two can never diverge. Accepts a streaming or batch DataFrame of
    bronze.logs rows."""
    # 1) Parse: strip a trailing ' |EXTRA|' junk suffix, then capture the
    #    common head (who/when/what/result) plus bytes and the free-form
    #    tail. Lines that don't match leave every field NULL (extract()
    #    maps the no-match "" to NULL) and are dropped by the expectations
    #    on logs_clean_stream below.
    parsed = bronze.select(
        # log_id hashes the ORIGINAL raw line (not the stripped one), so
        # junk-stripping and the fallback variants don't change dedup or
        # idempotency. Duplicate copies of the same line in one file merge
        # into one row.
        F.sha2(F.concat_ws("|", "source_file", "log_line"), 256).alias("log_id"),
        F.regexp_replace("log_line", r" \|EXTRA\|$", "").alias("stripped"),
        F.col("source_file"),
        F.col("updated_at"),
    ).select(
        "log_id",
        extract("client_ip", 1),
        # '-' = anonymous request
        extract("auth_user", 3),
        extract("ts_raw", 4),
        extract("method", 5),
        extract("req_target", 6),
        extract("protocol", 7),
        extract("status", 8),
        extract("bytes", 10),
        extract("tail", 11),
        "source_file",
        "updated_at",
    )

    # 2) Clean + standardize: '-' placeholder -> NULL, explicit casts keep
    #    the silver schema stable, email canonical lowercase to match
    #    customers, request target split into endpoint + query string.
    #    try_to_timestamp and the digits-only bytes guard never throw
    #    under ANSI mode: a bad value becomes NULL and the row is dropped
    #    by the expectations instead of killing the stream.
    cleaned = parsed.select(
        "log_id",
        "client_ip",
        # canonical lowercase email (matches silver.customers), NULL if '-'
        null_if_absent(F.lower(F.trim("auth_user"))).alias("auth_user"),
        # parse event time including the line's own UTC offset (-0300)
        F.try_to_timestamp("ts_raw", F.lit("dd/MMM/yyyy:HH:mm:ss Z")).alias("event_time"),
        "method",
        # "/products?sid=x" -> endpoint "/products" + query string "sid=x"
        F.regexp_extract("req_target", r"^[^?]*", 0).alias("endpoint"),
        F.when(F.regexp_extract("req_target", r"\?(.*)$", 1) != "",
               F.regexp_extract("req_target", r"\?(.*)$", 1)).alias("query_string"),
        "protocol",
        F.col("status").cast("int").alias("status_code"),
        # '-' = no response body (e.g. 204/304 responses); missing entirely
        # for 2 recovered lines -> NULL
        F.when(F.col("bytes").rlike(r"^[0-9]+$"), F.col("bytes")).cast("long").alias("response_bytes"),
        # referer / user_agent from the tail, for both the quoted and the
        # unquoted line variants (see tail_* helpers)
        null_if_absent(tail_referer(F.col("tail"))).alias("referer"),
        null_if_absent(tail_user_agent(F.col("tail"))).alias("user_agent"),
        "source_file",
        "updated_at",
    )
    return cleaned


@dp.temporary_view(name="logs_clean_stream")
# Data quality validation.
# Drop-level: blank/truncated/garbage lines do not parse, so every captured
# field is NULL -> the row is dropped from silver. expect_or_drop discards
# bad rows and keeps only a drop count (same behavior as the products
# file); the raw lines remain in bronze for inspection, and the rejects
# are captured in quarantine.logs_silver below.
@dp.expect_or_drop("client_ip_not_null", "client_ip IS NOT NULL")
@dp.expect_or_drop("event_time_not_null", "event_time IS NOT NULL")
@dp.expect_or_drop("method_valid", "method IN ('GET', 'HEAD', 'POST', 'PUT', 'DELETE', 'CONNECT', 'OPTIONS', 'TRACE', 'PATCH')")
@dp.expect_or_drop("status_code_valid", "status_code BETWEEN 100 AND 599")
# Referential integrity (warn): an authenticated request carries a customer
# email that must resolve to a current silver.customers row; anonymous
# requests are exempt. A customer ingested after their log lines shows as a
# false orphan until the next refresh (warn only, harmless).
@dp.expect("auth_user_in_customers", "auth_user IS NULL OR _customer_email_match IS NOT NULL")
def logs_clean_stream():
    # bronze.logs is append-only: every new file appends new rows.
    bronze = spark.readStream.table(BRONZE_LOGS)
    cleaned = cleaned_logs(bronze)
    # 3) Referential integrity: authenticated requests carry a customer
    #    email; left join the distinct current silver.customers emails
    #    (SCD Type 2 current rows) to verify it resolves. Caveat: a customer
    #    ingested after their log lines shows as a false orphan until the
    #    next refresh (warn only, harmless).
    customer_emails = (
        spark.read.table(SILVER_CUSTOMERS)
        .filter("__END_AT IS NULL")  # current version only
        .select(F.lower(F.trim("email")).alias("_customer_email_match"))
        .distinct()
    )
    return cleaned.join(
        customer_emails,
        cleaned.auth_user == customer_emails._customer_email_match,
        "left",
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.logs
#    Rows are MERGEd by log_id (hash of the raw line + file). Duplicate
#    lines collapse into one row and re-ingestion is idempotent.
#    SCD Type 1: events are immutable, only the latest copy is kept.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="logs",
    comment=(
        "Silver web access logs: parsed from Apache/Nginx combined format, "
        "cleaned ('-' -> NULL) and standardized; one row per unique log "
        "event (log_id), deduplicated via AUTO CDC merge"
    ),
)

dp.create_auto_cdc_flow(
    target="logs",
    source="logs_clean_stream",
    # MERGE on the line hash: duplicate lines (2.3k extra rows spread across
    # 640 of 739 files - verified double-writes, not one reloaded file)
    # collapse to one row. Caveats: identical same-second requests are
    # treated as double-writes; the hash includes source_file, so re-sending
    # the same file path is idempotent, but a renamed copy re-processes.
    keys=["log_id"],
    sequence_by="updated_at",
    # SCD Type 1 (default): immutable events, no history needed.
    # The FK match flag is a validation helper, not silver data.
    except_column_list=["_customer_email_match"],
)


# ---------------------------------------------------------------------------
# 3) Quarantine: b2b_ecommerce.quarantine.logs_silver
#    Every row excluded from silver.logs lands here instead of vanishing:
#    one append-only row per offending bronze row (duplicate copies
#    included), parsed to the same schema as silver.logs plus a
#    failed_checks column naming the drop rules it violated. Raw lines
#    stay in bronze.logs; this table is the review/worklist.
#    Reads bronze directly, NOT logs_clean_stream: the expect_or_drop
#    decorators above would drop the bad rows before they could reach
#    here.
# ---------------------------------------------------------------------------
@dp.table(
    # fully-qualified name: publishes outside the pipeline's silver schema
    # into the dedicated quarantine schema, as <table>_<layer>
    name="b2b_ecommerce.quarantine.logs_silver",
    comment=(
        "Log lines rejected by silver.logs data quality rules, kept for "
        "inspection and rework; failed_checks lists the violated rules"
    ),
)
def logs_quarantine():
    # Same rules as the drop-level expectations on logs_clean_stream, one
    # name per predicate. NOT (... IS TRUE) mirrors expect_or_drop exactly:
    # a NULL predicate (e.g. status_code NULL because the line never
    # parsed) counts as a violation.
    drop_checks = [
        ("client_ip_not_null", "client_ip IS NOT NULL"),
        ("event_time_not_null", "event_time IS NOT NULL"),
        ("method_valid", "method IN ('GET', 'HEAD', 'POST', 'PUT', 'DELETE', 'CONNECT', 'OPTIONS', 'TRACE', 'PATCH')"),
        ("status_code_valid", "status_code BETWEEN 100 AND 599"),
    ]
    return (
        cleaned_logs(spark.readStream.table(BRONZE_LOGS))
        .withColumn(
            "failed_checks",
            F.concat_ws(
                ", ",
                *[
                    F.when(F.expr(f"NOT (({predicate}) IS TRUE)"), F.lit(name))
                    for name, predicate in drop_checks
                ],
            ),
        )
        .filter(
            "NOT ("
            + " AND ".join(f"({predicate}) IS TRUE" for _, predicate in drop_checks)
            + ")"
        )
    )